In [4]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [16]:
def conv_block(in_channels:int  , out_channels:int , num_convs:int): 
    """VGG块儿
    Args:
        in_channels : int 输入通道数量
        out_channels: int 输出通达数量
        num_convs   : int 卷积层的层数
    Returns:
        layers : nn.Sequential 卷积层序列
    """
    layers = []
    for _ in range(num_convs):
        layers += [
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ]
        in_channels = out_channels
    layers.append(
        nn.MaxPool2d(
            kernel_size=2 , 
            stride=2
        )
    )
    return nn.Sequential(*layers)


In [17]:
print(conv_block(3 , 64 , 15))

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU(inplace=True)
  (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (7): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (8): ReLU(inplace=True)
  (9): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): ReLU(inplace=True)
  (12): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (13): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (14): ReLU(inplace=

In [18]:
class VGGNet_16(nn.Module):
    """VGG网络架构
    """
    def __init__(self , num_classes):
        super(VGGNet_16 , self).__init__()
        self.pool1 = conv_block(3 , 64 , 2)
        self.pool2 = conv_block(64 , 128 , 2)
        self.pool3 = conv_block(128 , 256 , 3)
        self.pool4 = conv_block(256 , 512 , 3)
        self.pool5 = conv_block(512 , 512 , 3)
        self._init_weights()# 开明初始化

    def forward(self, x):
        x = self.pool1(x)
        x = self.pool2(x)
        p3 = self.pool3(x)
        p4 = self.pool4(p3)
        p5 = self.pool5(p4)
        return p3 , p4 , p5
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m , nn.Conv2d):
                nn.init.kaiming_normal_( # 卷积的权重举证用凯明初始化
                    m.weight,
                    mode='fan_out',
                    nonlinearity='relu'
                )
            elif isinstance(m , nn.BatchNorm2d):# 非卷积的权重和偏置使用0进行填充
                nn.init.constant_(m.weight , 1)
                nn.init.constant_(m.bias , 0)

In [19]:
model = VGGNet_16(num_classes=20)
x = torch.randn(1 , 3 , 512 , 512)
p3 , p4 , p5 = model(x)
p3 , p4 , p5

(tensor([[[[0.2675, 0.0000, 0.0000,  ..., 0.0000, 0.5389, 0.8135],
           [1.0995, 0.0000, 2.7766,  ..., 1.6435, 1.3323, 1.0835],
           [1.5041, 0.5879, 1.1882,  ..., 1.1151, 0.6638, 1.7362],
           ...,
           [0.2786, 0.9833, 1.9640,  ..., 2.7619, 1.1332, 1.6162],
           [1.3980, 0.0000, 0.9246,  ..., 0.0000, 0.5935, 1.1055],
           [0.6261, 0.4717, 0.0000,  ..., 1.5894, 1.6354, 0.6028]],
 
          [[1.3566, 1.3957, 1.1478,  ..., 0.9178, 0.6114, 0.0000],
           [1.8647, 2.1613, 0.3633,  ..., 1.0773, 1.4068, 1.4736],
           [0.7811, 2.2926, 0.0000,  ..., 0.0000, 1.1168, 0.0000],
           ...,
           [1.6361, 1.3400, 1.8492,  ..., 0.1947, 0.6398, 1.1222],
           [0.8782, 0.3659, 0.0671,  ..., 0.8642, 1.9224, 1.1221],
           [0.5923, 1.7568, 1.1971,  ..., 2.7240, 1.5102, 0.5702]],
 
          [[1.4421, 0.7291, 1.4278,  ..., 1.1327, 0.8403, 1.1589],
           [1.6430, 0.9288, 1.2924,  ..., 0.3016, 1.6075, 0.0000],
           [1.5033, 1.09